# **3. 현대 CNN과 전이학습** — 포도 잎 병해 진단


---


- 실무 상황
    - "농장에서 찍은 포도 잎 사진으로 병해를 판별하고 싶은데, 라벨 붙은 사진은 수백 장뿐이다."
- 목표
    - ① 처음부터 학습(scratch) vs **특징 추출(feature extraction)** vs **미세조정(fine-tuning)** 비교
    - ② **Grad-CAM** 으로 모델이 본 곳 확인
    - ③ 실험실 데이터 → 현장 데이터의 **도메인 차이** 체험
    - ④ AI Hub 시설포도 실데이터로 확장
- 데이터
    - 데이터 A (공통)
        - PlantVillage 포도 잎 4클래스 (Hughes & Salathé, 2015, arXiv:1511.08060)
        — GitHub에서 자동 다운로드(약 80MB)
    - 데이터 B (확장)
        - AI Hub「식물 병 유발 통합 데이터」시설포도 (NIA, 2022)
        — **본인이 신청·다운로드한 PC에서 실행**
    - 데이터 C (선택)
        - Hugging Face `beans` (콩잎 3클래스, MIT 라이선스)
- 실행 환경 | Colab 또는 PC,  (GPU 있으면 자동 사용)



---


**(코랩에서)한글 폰트 지정하는 방법**

In [ ]:
# 코랩에서 실행
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import shutil
import warnings
import logging

# ── 1. 나눔 폰트 설치 ──────────────────────────────────────────
!apt-get install -y fonts-nanum > /dev/null 2>&1

# ── 2. matplotlib 캐시 디렉터리 삭제 ──────────────────────────
shutil.rmtree(mpl.get_cachedir(), ignore_errors=True)

# ── 3. 폰트 등록 및 유니코드 마이너스 경고 방지 ──────────────────
font_path = '/usr/share/fonts/truetype/nanum/NanumBarunGothic.ttf'
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams['font.family'] = font_name
plt.rcParams['font.size'] = 12
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (6, 4)

# 수학 기호(마이너스 등)를 일반 텍스트로 처리하도록 설정하여 폰트 에러 차단
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.default'] = 'regular'

# mathtext 및 font_manager 관련 폰트 결함 경고 완전히 무시 설정
logging.getLogger('matplotlib.mathtext').setLevel(logging.ERROR)
logging.getLogger('matplotlib.font_manager').setLevel(logging.ERROR)
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib')
warnings.filterwarnings('ignore', category=UserWarning, module='matplotlib.font_manager')



---


## 0. 환경 설정

In [ ]:
# [Colab 명령] 한글 폰트: !apt-get -qq install -y fonts-nanum > /dev/null ; 이후 세션 다시 시작
import os, time, glob, json, random, subprocess
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

def set_korean_font():
    installed = {f.name for f in fm.fontManager.ttflist}
    for name in ["Malgun Gothic", "NanumGothic", "AppleGothic", "Noto Sans CJK KR"]:
        if name in installed:
            plt.rcParams["font.family"] = name; break
    plt.rcParams["axes.unicode_minus"] = False

set_korean_font()
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| device =", device)



---



## **3-1. 실습 데이터 A — PlantVillage 포도 잎 4클래스**

전체 저장소(수 GB) 대신 **포도 폴더 4개만** `git sparse-checkout`으로 받습니다.

| 클래스 | 의미 |
|---|---|
| Black_rot | 검은썩음병 |
| Esca | 에스카병(Black Measles) |
| Leaf_blight | 잎마름병(Isariopsis Leaf Spot) |
| healthy | 정상 |

In [ ]:
PV_DIR = "./data/plantvillage_repo"
GRAPE = ["Grape___Black_rot", "Grape___Esca_(Black_Measles)",
         "Grape___Leaf_blight_(Isariopsis_Leaf_Spot)", "Grape___healthy"]
CLASSES = ["Black_rot", "Esca", "Leaf_blight", "healthy"]

if not os.path.exists(os.path.join(PV_DIR, "raw", "color", GRAPE[0])):
    os.makedirs("./data", exist_ok=True)
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/spMohanty/PlantVillage-Dataset", PV_DIR], check=True)
    subprocess.run(["git", "-C", PV_DIR, "sparse-checkout", "set"] + [f"raw/color/{g}" for g in GRAPE], check=True)

all_items = []
for ci, g in enumerate(GRAPE):
    files = sorted(glob.glob(os.path.join(PV_DIR, "raw", "color", g, "*")))
    print(f"{CLASSES[ci]:<12} {len(files):>5}장")
    all_items += [(f, ci) for f in files]

### **'적은 데이터'로 만들기**

실제 현장에서는 라벨 데이터가 적습니다. 클래스마다 **학습 60 / 검증 30 / 테스트 50장**만 씁니다.

In [ ]:
N_TRAIN, N_VAL, N_TEST = 60, 30, 50
rng = random.Random(SEED)
splits = {"train": [], "val": [], "test": []}
for ci in range(len(CLASSES)):
    items = [it for it in all_items if it[1] == ci]
    rng.shuffle(items)
    splits["train"] += items[:N_TRAIN]
    splits["val"]   += items[N_TRAIN:N_TRAIN + N_VAL]
    splits["test"]  += items[N_TRAIN + N_VAL:N_TRAIN + N_VAL + N_TEST]
print({k: len(v) for k, v in splits.items()})

IMG = 160                                     # CPU 속도를 위해 224 대신 160 사용
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]   # ImageNet 통계 (사전학습 모델과 맞춤)
tf_train = transforms.Compose([transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0)),
                               transforms.RandomHorizontalFlip(), transforms.ToTensor(),
                               transforms.Normalize(MEAN, STD)])
tf_eval = transforms.Compose([transforms.Resize(IMG), transforms.CenterCrop(IMG),
                              transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class ImageListDataset(Dataset):
    """(파일경로, 라벨) 리스트를 받는 범용 Dataset. crop 상자(x,y,w,h)가 있으면 잘라서 사용."""
    def __init__(self, items, tf):
        self.items, self.tf = items, tf
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, label, *box = self.items[i]
        img = Image.open(path).convert("RGB")
        if box and box[0] is not None:
            x, y, w, h = box[0]; img = img.crop((x, y, x + w, y + h))
        return self.tf(img), label

loaders = {
    "train": DataLoader(ImageListDataset(splits["train"], tf_train), batch_size=32, shuffle=True),
    "val":   DataLoader(ImageListDataset(splits["val"], tf_eval), batch_size=64),
    "test":  DataLoader(ImageListDataset(splits["test"], tf_eval), batch_size=64),
}

fig, axes = plt.subplots(1, 8, figsize=(15, 2.4))
for k, ax in enumerate(axes):
    p, c = splits["test"][k * 25]
    ax.imshow(Image.open(p))
    ax.set_title(CLASSES[c], fontsize=9)
    ax.axis("off")
plt.suptitle("PlantVillage 포도 잎 예시 (실험실처럼 균일한 배경)")
plt.tight_layout()
plt.show()



---



## **3-2. 공통 학습 함수**

In [ ]:
def run_epoch(model, loader, opt=None):
    train = opt is not None
    model.train(train)
    n, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = F.cross_entropy(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(y); correct += (out.argmax(1) == y).sum().item(); n += len(y)
    return loss_sum / n, correct / n

def fit(model, epochs, lr, tag):
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    t0 = time.time()
    for ep in range(1, epochs + 1):
        tr_loss, tr_acc = run_epoch(model, loaders["train"], opt)
        _, va_acc = run_epoch(model, loaders["val"])
        print(f"  [{tag}] epoch {ep}  train_acc {tr_acc:.3f}  val_acc {va_acc:.3f}")
    sec = time.time() - t0
    _, te_acc = run_epoch(model, loaders["test"])
    return dict(test_acc=te_acc, sec=sec, trainable=sum(p.numel() for p in params))



---



## **3-3. 실험: 사전학습의 힘 확인**

- **전이학습(Transfer Learning)**
    - 대규모 데이터로 미리 학습한 AI 모델의 지식(가중치)/보는 법을 새로운 문제에 재활용하여, 적은 데이터와 학습 시간으로 효율적으로 모델을 학습시키는 기법입니다

### **사전학습 ResNet18 불러오기** (ImageNet 1,000클래스, 약 128만 장으로 학습됨)

기본은 torchvision 공식 가중치(`ResNet18_Weights.DEFAULT`)입니다. 방화벽 등으로 실패하면
GitHub에 공개된 timm 가중치(ResNet strikes back A1 레시피, Wightman et al. 2021)를 대신 씁니다. 두 가중치는 **구조가 같아 그대로 호환**됩니다.

In [ ]:
TIMM_R18 = ("https://github.com/huggingface/pytorch-image-models/releases/download/"
            "v0.1-rsb-weights/resnet18_a1_0-d63eafa0.pth")

def load_resnet18_pretrained():
    try:
        m = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        print("torchvision 공식 ImageNet 가중치 사용 (IMAGENET1K_V1)")
    except Exception as e:
        print("torchvision 가중치 다운로드 실패 → timm A1 가중치 사용:", type(e).__name__)
        m = models.resnet18()
        m.load_state_dict(torch.hub.load_state_dict_from_url(TIMM_R18, map_location="cpu", progress=False))
    return m

### **① 처음부터 학습 (scratch) — 작은 CNN**

In [ ]:
def small_cnn(n_classes=4):
    def block(ci, co):
        return [nn.Conv2d(ci, co, 3, padding=1), nn.BatchNorm2d(co), nn.ReLU(), nn.MaxPool2d(2)]
    return nn.Sequential(*block(3, 32), *block(32, 64), *block(64, 128), *block(128, 128),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(128, n_classes))

results = {}
torch.manual_seed(SEED)
scratch = small_cnn()
results["① scratch 작은 CNN"] = fit(scratch, epochs=8, lr=3e-3, tag="scratch")

### **② 특징 추출 (Feature Extraction)** — 몸통은 얼리고(freeze) 마지막 층만 교체·학습

In [ ]:
torch.manual_seed(SEED)
fe = load_resnet18_pretrained()
for p in fe.parameters():
    p.requires_grad = False                         # 모든 층 동결
fe.fc = nn.Linear(fe.fc.in_features, len(CLASSES))  # 새 분류기 (512 → 4), 이 층만 학습
results["② 특징 추출 (fc만 학습)"] = fit(fe, epochs=4, lr=3e-3, tag="feature-extract")

### **③ 미세조정 (Fine-tuning)** — 마지막 블록(layer4)까지 풀어서 작은 학습률로

In [ ]:
torch.manual_seed(SEED)
ft = load_resnet18_pretrained()
for name, p in ft.named_parameters():
    p.requires_grad = name.startswith("layer4")      # layer4만 학습 허용
ft.fc = nn.Linear(ft.fc.in_features, len(CLASSES))   # 새 fc는 기본 requires_grad=True
results["③ 미세조정 (layer4+fc)"] = fit(ft, epochs=4, lr=5e-4, tag="fine-tune")

In [ ]:
print(f"\n{'방법':<22}{'학습 파라미터':>14}{'테스트 정확도':>14}{'학습 시간(초)':>14}")
for k, r in results.items():
    print(f"{k:<22}{r['trainable']:>14,}{r['test_acc']:>14.3f}{r['sec']:>14.1f}")

> ★ 참고 : 결과는 시드·하드웨어에 따라 조금씩 달라집니다. 핵심은 **같은 적은 데이터에서 사전학습 모델이 훨씬 빨리, 더 높은 정확도에 도달**한다는 점입니다.

## **3-4. Grad-CAM** — 모델은 잎의 어디를 보고 판단했을까? (Selvaraju et al., 2017)

- **Grad-CAM(Gradient-weighted Class Activation Mapping)**
    - CNN이 이미지의 어떤 영역을 근거로 특정 클래스를 예측했는지 기울기(Gradient)를 활용해 히트맵(Heatmap)으로 시각화하는 설명 가능한 AI(XAI) 기법입니다

| 단계 | 계산 |
|---|---|
| ① | 마지막 합성곱 블록(layer4)의 특징맵 $A^k$ 저장 (forward hook) |
| ② | 예측 클래스 점수 $y^c$ 의 기울기 $\partial y^c/\partial A^k$ 저장 (backward hook) |
| ③ | 채널 중요도 $\alpha_k = $ 기울기의 공간 평균 (GAP) |
| ④ | $\text{CAM} = \text{ReLU}(\sum_k \alpha_k A^k)$ → 입력 크기로 확대해 겹쳐 그리기 |

In [ ]:
class GradCAM:
    def __init__(self, model, layer):
        self.model, self.acts, self.grads = model.eval(), None, None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "acts", o))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "grads", go[0]))
    def __call__(self, x, cls=None):
        for p in self.model.parameters():
            p.requires_grad_(True)                       # 동결된 층도 기울기 흐르게
        out = self.model(x)
        cls = out.argmax(1).item() if cls is None else cls
        self.model.zero_grad()
        out[0, cls].backward()
        alpha = self.grads.mean(dim=(2, 3), keepdim=True)          # ③ 채널 중요도
        cam = F.relu((alpha * self.acts).sum(1, keepdim=True))     # ④
        cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.detach().cpu().numpy(), cls, out.softmax(1)[0, cls].item()

def denorm(t):
    return (t.permute(1, 2, 0).numpy() * STD + MEAN).clip(0, 1)

cam_model = ft.to(device)
gradcam = GradCAM(cam_model, cam_model.layer4)
test_ds = ImageListDataset(splits["test"], tf_eval)
fig, axes = plt.subplots(2, 8, figsize=(16, 5.0))
for j, ci in enumerate(range(len(CLASSES))):
    for r in range(2):
        idx = ci * N_TEST + r
        x, y = test_ds[idx]
        cam, pred, prob = gradcam(x[None].to(device))
        col = j * 2 + r
        axes[0, col].imshow(denorm(x))
        axes[0, col].set_title(f"정답 {CLASSES[y]}", fontsize=8)
        axes[1, col].imshow(denorm(x))
        axes[1, col].imshow(cam, cmap="jet", alpha=0.45)
        axes[1, col].set_title(f"예측 {CLASSES[pred]} ({prob:.2f})", fontsize=8,
                               color="blue" if pred == y else "red")
for ax in axes.flat: ax.axis("off")
plt.suptitle("Grad-CAM: 미세조정 ResNet18이 근거로 삼은 영역 (빨강=중요)"); plt.tight_layout(); plt.show()

## 3-5. **도메인 차이**(domain shift) 체험: 실험실 → 현장

PlantVillage는 **균일한 배경**에서 찍은 사진입니다. Mohanty et al.(2016)은 이 데이터에서 99%대 정확도를 보고했지만,
다른 출처의 사진에서는 정확도가 크게 떨어진다고 함께 보고했습니다.
아래에서는 현장 촬영 조건(흐림·조명·기울기·저해상도)을 흉내 낸 변환으로 같은 테스트셋을 다시 평가합니다.

### **① 현장 촬영 조건 테스트셋으로 다시 평가**
(흐림·조명·기울기·저해상도)을 흉내 낸 변환으로 같은 테스트셋을 다시 평가

In [ ]:
tf_field = transforms.Compose([
    transforms.Resize(IMG), transforms.CenterCrop(IMG),
    transforms.RandomPerspective(distortion_scale=0.3, p=1.0),             # 비스듬한 촬영
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.04),  # 조명
    transforms.GaussianBlur(5, sigma=(0.8, 1.6)),                          # 초점 흐림
    transforms.Resize(96), transforms.Resize(IMG),                         # 저해상도
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
torch.manual_seed(SEED)
field_loader = DataLoader(ImageListDataset(splits["test"], tf_field), batch_size=64)
for name, m in [("① scratch", scratch), ("② 특징 추출", fe), ("③ 미세조정", ft)]:
    torch.manual_seed(SEED)
    _, acc_clean = run_epoch(m, loaders["test"])
    _, acc_field = run_epoch(m, field_loader)
    print(f"{name}: 깨끗한 테스트 {acc_clean:.3f}  →  현장 흉내 테스트 {acc_field:.3f}")

x_demo, _ = ImageListDataset(splits["test"][:1], tf_field)[0]
x_clean, _ = ImageListDataset(splits["test"][:1], tf_eval)[0]
fig, axes = plt.subplots(1, 2, figsize=(5, 2.6))
axes[0].imshow(denorm(x_clean))
axes[0].set_title("원본")
axes[1].imshow(denorm(x_demo))
axes[1].set_title("현장 흉내 변환")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

### **② 대응: 현장 조건을 반영한 증강** :  '학습 증강'에 넣어 다시 미세조정

학습 데이터에 같은 종류의 변형(원근·조명·흐림·저해상도)을 **확률 50%** 로 섞어 학습하면 어떻게 될까요?

In [ ]:
tf_train_robust = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.7, 1.0)), transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.RandomPerspective(0.3, p=1.0),
                            transforms.ColorJitter(0.4, 0.4, 0.4, 0.04),
                            transforms.GaussianBlur(5, sigma=(0.8, 1.6)),
                            transforms.Resize(96), transforms.Resize(IMG)], p=0.5),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
loaders["train"] = DataLoader(ImageListDataset(splits["train"], tf_train_robust), batch_size=32, shuffle=True)
torch.manual_seed(SEED)
ft_robust = load_resnet18_pretrained()
for name, p in ft_robust.named_parameters():
    p.requires_grad = name.startswith("layer4")
ft_robust.fc = nn.Linear(ft_robust.fc.in_features, len(CLASSES))
r_rb = fit(ft_robust, epochs=6, lr=5e-4, tag="fine-tune + 현장 증강")
torch.manual_seed(SEED)
_, acc_field_rb = run_epoch(ft_robust, field_loader)
print(f"④ 미세조정 + 현장 증강: 깨끗한 테스트 {r_rb['test_acc']:.3f}  →  현장 흉내 테스트 {acc_field_rb:.3f}")
loaders["train"] = DataLoader(ImageListDataset(splits["train"], tf_train), batch_size=32, shuffle=True)  # 원래대로

**실무 교훈** : 테스트 정확도 99%라도 **학습 데이터와 다른 환경**에서는 성능이 무너질 수 있습니다.
→ 현장 데이터(예: AI Hub 시설포도) 수집 · 증강 · 현장 검증셋이 필수입니다.